# 02 — vLLM Offline Inference

Goal: run the same prompts using vLLM offline inference through the `LLM` class and compare against the Transformers baseline.

In [ ]:
import sys, time
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(repo_root / 'src'))

import pandas as pd
from vllm import LLM, SamplingParams

from vllm_lab.utils import load_config, get_gpu_snapshot

config = load_config(repo_root / 'configs' / 'lab_config.yaml')
model_name = config['model']['default_name']
prompts = config['benchmark']['prompts']
max_new_tokens = config['model']['max_new_tokens']
print('gpu:', get_gpu_snapshot())

vLLM will manage batching and KV cache internally. The first run may include model loading and warm-up costs, so benchmark generation after the engine is created.

In [ ]:
llm = LLM(
    model=model_name,
    dtype='auto',
    max_model_len=2048,
    gpu_memory_utilization=0.85,
)
params = SamplingParams(temperature=0.0, max_tokens=max_new_tokens)

In [ ]:
# Warm-up run: reduces one-off startup noise.
_ = llm.generate(['Say hello in one sentence.'], params)

In [ ]:
start = time.perf_counter()
outputs = llm.generate(prompts, params)
wall_s = time.perf_counter() - start

rows = []
for prompt, output in zip(prompts, outputs):
    text = output.outputs[0].text
    output_tokens = len(output.outputs[0].token_ids)
    prompt_tokens = len(output.prompt_token_ids)
    rows.append({
        'backend': 'vllm_offline_batch',
        'prompt': prompt,
        'output_text': text,
        'prompt_tokens': prompt_tokens,
        'output_tokens': output_tokens,
        'batch_wall_time_s': wall_s,
        'aggregate_output_tokens_per_second': sum(r.get('output_tokens', 0) for r in rows) / wall_s if rows else None,
    })

df = pd.DataFrame(rows)
df['aggregate_output_tokens_per_second'] = df['output_tokens'].sum() / wall_s
df

In [ ]:
out = repo_root / 'results' / 'vllm_offline_inference.csv'
df.to_csv(out, index=False)
print('wall_s:', wall_s)
print('aggregate output tokens/s:', df['output_tokens'].sum() / wall_s)
print('wrote', out)

Compare with Notebook 01. Be careful: small prompt counts may exaggerate noise. Repeat with more prompts before drawing conclusions.